In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# 1. 환경 변수 로드
load_dotenv()

# 2. 아주 가벼운 모델로 최소한의 호출 테스트
test_llm = ChatOpenAI(model="gpt-4.1-mini", max_tokens=10)

try:
    print("📡 API 연결 테스트 시작...")
    response = test_llm.invoke("안녕? 대답할 수 있다면 'OK'라고만 해줘.")
    print(f"✅ 연결 성공! 응답: {response.content}")
except Exception as e:
    print("❌ 연결 실패!")
    print(f"에러 메시지: {e}")

📡 API 연결 테스트 시작...
❌ 연결 실패!
에러 메시지: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


# State 정의
-> example structure는 추후에 노드에서 structured output으로 넣도록 하면 됨.

In [ ]:
from typing import TypedDict, List, Dict, Optional, Annotated
import operator

class AgentState(TypedDict):
    # 1. 입력 컨텍스트
    user_id: int
    location_name: str
    raw_category: str       # 카카오 API 원본 문자열
    category_id: int        # SQL 조회를 위해 매핑된 DB 카테고리 ID
    
    # 2. DB Node가 SQL로 빌드한 상세 컨텍스트 (Candidate Cards)
    candidate_cards: List[dict] 
    """
    Example Structure:
    [
      {
        "user_card_id": 101,
        "card_name": "KB 토심이",
        "last_month_spent": 450000,
        "remaining_limit": 500,
        "tier_thresholds": [
            {"name": "1구간", "amount": 300000},
            {"name": "2구간", "amount": 600000}
        ],
        "benefit_hint": "스타벅스 10% 청구할인"
      }
    ]
    """
    
    # 3. 실시간 현장 결제 이벤트 (Offline Events)
    offline_events: Annotated[List[dict], operator.add]
    """
    Example Structure:
    [
        {
            "brand": "스타벅스", 
            "pay_system": "Npay", 
            "benefit_detail": "50% 적립", 
            "max_benefit": 3000, 
            "d_day": "D-2"
        }
    ]
    """
    
    # 4. RAG Workers가 PDF를 보고 판별한 결과
    rag_analysis: Annotated[List[dict], operator.add]
    """
    Example Structure:
    [
        {
            "card_id": 101,
            "card_name": "KB 토심이",
            "detected_tier": "1구간",
            "benefit_type": "DISCOUNT",
            "benefit_detail": "스타벅스 2,000원 할인",
            "calculated_benefit": 500,
            "hurdle": 10000,
            "is_excluded": False,  # 파이썬에서는 False로 작성해야 합니다.
            "caveats": "사이렌 오더 결제 시 제외, 백화점 입점 매장 제외",
            "evidence": "PDF 4페이지 '스타벅스 혜택' 섹션..."
        }
    ]
    """
    
    # 5. 최종 출력
    final_advice: str

## Node / Tool / Router

In [ ]:
import requests
from bs4 import BeautifulSoup

# 1. 카드고릴라에 API 요청
card_id = "2929"
url = f"https://api.card-gorilla.com:8080/v1/cards/{card_id}"

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Origin": "https://www.card-gorilla.com",
    "Referer": "https://www.card-gorilla.com/",
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    data = response.json()
    
    # 카드 이름 출력
    card_name = data.get("name")
    print(f"✅ 카드명: {card_name}\n")
    
    # 주요 혜택 정제해서 출력
    for benefit in data.get("key_benefit", []):
        title = benefit.get("title")
        raw_info = benefit.get("info", "")
        
        # HTML 태그 제거 및 줄바꿈 정리
        soup = BeautifulSoup(raw_info, "html.parser")
        clean_info = soup.get_text(separator="\n").strip()
        
        print(f"[{title}]")
        print(clean_info)
        print("-" * 30)
else:
    print(f"❌ 실패! 상태 코드: {response.status_code}")
    print("응답 내용:", response.text[:200])